# FASE 1: MEMUAT DATA
Bagian ini berfokus pada persiapan awal lingkungan kerja, pengaturan pustaka yang dibutuhkan, dan pemuatan berbagai dataset (data titik panas, data iklim, peta provinsi, dan data sekolah) ke dalam memori.

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)
np.random.seed(42)

DATA_RAW = r'd:\Code\Fireline\data\raw'
DATA_PROCESSED = r'd:\Code\Fireline\data\processed'
OUTPUTS = r'd:\Code\Fireline\outputs\figures'

## 1. Memuat Data Utama (FIRMS Hotspot)
Kita mulai dengan memuat data titik api satelit dari file CSV. 
Setelah dimuat, kita akan lihat ukuran datanya dan melihat beberapa baris pertama agar ada gambaran seperti apa data aslinya sebelum kita bersihkan.

In [ ]:
from IPython.display import display, Markdown

df_hotspot = pd.read_csv(os.path.join(DATA_RAW, 'Fireline_hotspot_kalimantan_viirs_noaa20_2024_2026.csv'))
display(Markdown(f"**Ukuran data awal:** {df_hotspot.shape[0]:,} baris dan {df_hotspot.shape[1]} kolom."))
display(df_hotspot.head())

Setelah melihat data mentahnya, kita cek apakah ada data yang kosong (null). Data kosong bisa bikin error di perhitungan nanti, jadi harus dipastikan bersih.

In [ ]:
null_counts = df_hotspot.isnull().sum()
if null_counts.sum() > 0:
    display(Markdown(f"**Total data kosong:** {null_counts.sum()}"))
else:
    display(Markdown("**Tidak ada nilai kosong di dataset.**"))

print("\n[1.5] Jumlah nilai kosong per kolom:")
null_counts = df_hotspot.isnull().sum()
print(null_counts)
if null_counts.sum() > 0:
    print(f"\n  Total nilai kosong: {null_counts.sum()}")
    print(f"     Kolom dengan nilai kosong: {list(null_counts[null_counts > 0].index)}")
else:
    print("\n  Tidak ditemukan nilai kosong.")

print("\n[1.6] Pemeriksaan duplikasi:")
exact_dupes = df_hotspot.duplicated().sum()
print(f"     Duplikasi persis: {exact_dupes:,}")

print("\n[1.7] Statistik deskriptif untuk kolom numerik:")
desc = df_hotspot.describe()
print(desc)

print("\n[1.8] Jumlah nilai unik untuk kolom kategorikal:")
for col in ['confidence', 'satellite', 'daynight', 'type', 'instrument', 'version']:
    if col in df_hotspot.columns:
        print(f"\n  {col}:")
        print(f"  {df_hotspot[col].value_counts().to_dict()}")

## Memuat Dataset Iklim IDN (Climate Data Daily)
Kita memuat data historis iklim harian dan detail stasiun dari BMKG yang akan digunakan nanti untuk korelasi cuaca dan kemunculan api.

In [ ]:
print("\n[1.9] Memuat Climate Data Daily IDN")
df_climate = pd.read_csv(os.path.join(DATA_RAW, 'climate_data.csv'))
print(f"     Ukuran: {df_climate.shape}")
print(f"     Kolom: {list(df_climate.columns)}")
print(f"     Data teratas:")
print(df_climate.head(3))

df_station = pd.read_csv(os.path.join(DATA_RAW, 'station_detail.csv'))
print(f"\n     Ukuran detail stasiun: {df_station.shape}")
print(f"     Kolom: {list(df_station.columns)}")
print(df_station.head())

## Memuat Dataset Cuaca Pontianak
Data ini merupakan sampel cuaca harian dari satu wilayah spesifik (Pontianak) untuk kepentingan analisis tambahan.

In [ ]:
print("\n[1.10] Memuat data Pontianak Weather Daily")
df_pontianak = pd.read_csv(os.path.join(DATA_RAW, 'pontianak_weather_daily_2021_2024.csv'))
print(f"     Ukuran: {df_pontianak.shape}")
print(f"     Kolom: {list(df_pontianak.columns)}")
print(f"     Rentang tanggal: {df_pontianak.iloc[:, 0].min()} hingga {df_pontianak.iloc[:, 0].max()}")
print(df_pontianak.head(3))

## 2. Memuat Peta Provinsi Indonesia
Supaya nanti kita bisa tahu tiap titik api jatuhnya di provinsi mana, kita perlu data peta (poligon) batas provinsi. 
Kita pakai library Geopandas untuk memuat file GeoJSON-nya, lalu kita intip datanya.

In [ ]:
import geopandas as gpd

gdf_prov = gpd.read_file(os.path.join(DATA_RAW, 'indonesia-province-jml-penduduk.json'))
display(Markdown(f"**Peta provinsi berhasil dimuat.** Total provinsi: {len(gdf_prov)}"))
display(gdf_prov.head(2))

# Ambil khusus Kalimantan saja untuk basemap nanti
gdf_prov_kalimantan = gdf_prov[gdf_prov['Propinsi'].str.contains('KALIMANTAN', case=False, na=False)]

## Memuat Dataset Infrastruktur (Sekolah)
Data titik lokasi sekolah di Indonesia yang akan membantu kita dalam menentukan skor kerentanan dan paparan sosial terhadap kebakaran.

In [ ]:
print("\n[1.12] Memuat dataset sekolah Indonesia")
df_schools = pd.read_csv(os.path.join(DATA_RAW, 'complete_data.csv'))
print(f"     Ukuran: {df_schools.shape}")
print(f"     Kolom: {list(df_schools.columns)}")
has_coords = any(c.lower() in ['latitude', 'longitude', 'lat', 'lon', 'lng'] for c in df_schools.columns)
print(f"     Memiliki kolom koordinat: {has_coords}")
if has_coords:
    print("     Data level titik tersedia untuk penggabungan spasial yang lebih presisi")
else:
    print("     Tidak ada koordinat, hanya data agregat level provinsi")
print(df_schools.head(3))

## Ringkasan Inventaris Data
Seluruh dataset utama dan pendukung telah dimuat. Berikut adalah ringkasan jumlah baris atau elemen dari setiap dataset.

In [ ]:
print("\n[1.15] RINGKASAN INVENTARIS DATA")
print(f"FIRMS Hotspot Kalimantan: {df_hotspot.shape[0]:,} baris")
print(f"Climate Data Daily IDN: {df_climate.shape[0]:,} baris")
print(f"Detail Stasiun: {df_station.shape[0]:,} baris")
print(f"Cuaca Pontianak Harian: {df_pontianak.shape[0]:,} baris")
print(f"Peta Provinsi dan Populasi: {n_features if 'n_features' in dir() else '?'} fitur")
print(f"Dataset Sekolah: {df_schools.shape[0]:,} baris")

# FASE 2: BERSIH-BERSIH DATA (CLEANING)
Setelah semua data masuk, kita perlu periksa ulang. Apakah format waktunya sudah benar? Apakah ada data ganda? Di fase ini kita bersihkan data yang kotor agar analisis kita nanti valid.

## 1. Penyesuaian Zona Waktu
Data asli dari NASA FIRMS memakai waktu UTC. Karena kita menganalisis Kalimantan, kita ubah waktunya jadi WIB (UTC+7) supaya analisis "kebakaran siang vs malam" nanti tidak salah kaprah.

In [ ]:
expected_columns = ['latitude', 'longitude', 'acq_date', 'acq_time', 'confidence', 'frp', 'daynight', 'type', 'satellite', 'instrument', 'version', 'scan', 'track']
actual_columns = df_hotspot.columns.tolist()

for exp in expected_columns:
    if exp not in actual_columns:
        if exp == 'brightness' and 'bright_ti4' in actual_columns:
            print(f"     Kolom {exp} tidak ditemukan, tetapi ada bright_ti4")
        elif exp == 'bright_t31' and 'bright_ti5' in actual_columns:
            print(f"     Kolom {exp} tidak ditemukan, tetapi ada bright_ti5")
        else:
            print(f"     Kolom hilang: {exp}")

if 'brightness' in actual_columns:
    print("     Kolom bernama brightness ditemukan, interpretasi sebagai temperatur kecerahan dalam Kelvin.")

print("\n[2.2] Penanganan waktu konversi ke WIB")
df_hotspot['acq_date'] = pd.to_datetime(df_hotspot['acq_date'])

df_hotspot['acq_time_str'] = df_hotspot['acq_time'].astype(str).str.zfill(4)
df_hotspot['hour_utc'] = df_hotspot['acq_time_str'].str[:2].astype(int)
df_hotspot['minute_utc'] = df_hotspot['acq_time_str'].str[2:].astype(int)

df_hotspot['datetime_utc'] = pd.to_datetime(
    df_hotspot['acq_date'].dt.strftime('%Y-%m-%d') + ' ' + 
    df_hotspot['acq_time_str'].str[:2] + ':' + df_hotspot['acq_time_str'].str[2:],
    format='%Y-%m-%d %H:%M'
)

df_hotspot['datetime_wib'] = df_hotspot['datetime_utc'] + pd.Timedelta(hours=7)
df_hotspot['hour_local'] = df_hotspot['datetime_wib'].dt.hour
df_hotspot['date_local'] = df_hotspot['datetime_wib'].dt.date

print(f"     Rentang tanggal UTC: {df_hotspot['acq_date'].min()} hingga {df_hotspot['acq_date'].max()}")
print(f"     Rentang tanggal WIB: {df_hotspot['datetime_wib'].min()} hingga {df_hotspot['datetime_wib'].max()}")

## 2. Hapus Data Ganda dan Titik Bukan Vegetasi
Terkadang satelit mendeteksi hal yang sama dua kali. Kita hapus duplikatnya. 
Selain itu, satelit juga kadang mendeteksi panas dari pabrik atau gunung berapi (type != 0). Karena fokus kita karhutla, kita hanya simpan tipe 0 (kebakaran vegetasi/lahan).

In [ ]:
exact_dupes = df_hotspot.duplicated().sum()
display(Markdown(f"**Titik panas yang persis ganda (duplikat):** {exact_dupes:,} baris"))

key_cols = ['latitude', 'longitude', 'acq_date', 'acq_time']
near_dupes = df_hotspot.duplicated(subset=key_cols, keep=False).sum()
print(f"     Duplikat mirip lokasi dan waktu yang sama: {near_dupes:,}")

if exact_dupes > 0:
    print(f"     Menghapus {exact_dupes} duplikat persis")
    df_hotspot = df_hotspot.drop_duplicates().reset_index(drop=True)

print("\n[2.4] Penyaringan tipe data")
type_dist = df_hotspot['type'].value_counts()
print(f"     Distribusi tipe:")
for t, c in type_dist.items():
    label = {0: 'kebakaran vegetasi', 1: 'gunung berapi', 
             2: 'sumber darat statis lainnya', 3: 'lepas pantai'}.get(t, 'tidak diketahui')
    print(f"       tipe {t} {label}: {c:,} ({c/len(df_hotspot)*100:.1f} persen)")

non_veg_count = len(df_hotspot[df_hotspot['type'] != 0])
if non_veg_count > 0:
    print(f"     Mempertahankan tipe 0 saja dan menghapus {non_veg_count:,} deteksi non vegetasi")
    df_fire = df_hotspot[df_hotspot['type'] == 0].copy()
else:
    df_fire = df_hotspot.copy()

print("\n[2.5] Penyandian tingkat kepercayaan")
confidence_map = {'l': 1, 'n': 2, 'h': 3}
if df_fire['confidence'].dtype == object:
    df_fire['confidence_ord'] = df_fire['confidence'].map(confidence_map)
    print(f"     Tingkat kepercayaan berhasil dipetakan ke angka 1 2 3")

## 4. Cek Geografis (Filter Bounding Box)
Kita saring titik api agar hanya mencakup wilayah Kalimantan.

Batas kotak (bounding box) yang dipakai:
  Latitude : -4.5 s/d  4.5   (batas selatan mencakup Kalteng, batas utara +0.15° di atas titik paling utara
                               data aktual ~4.35° agar tidak ada observasi valid yang terbuang)
  Longitude: 108.0 s/d 119.5 (batas timur +0.01° di atas titik paling timur data aktual ~119.49°)

CATATAN: Batas ini sengaja dibuat sedikit lebih lebar dari batas administratif Kalimantan
(lat: -4.17 s/d 7.37, lon: 108.0 s/d 119.0) untuk memastikan TIDAK ADA observasi valid yang
terbuang hanya karena batas kotak terlalu ketat. Pada versi awal analisis, batas lat 3.5 dan
lon 117.5 menyebabkan ±3.798 titik valid di Kaltara dan Kaltim bagian timur ikut terbuang.
Visualisasi di bawah menunjukkan titik mana saja (jika ada) yang dibuang oleh filter ini.

In [ ]:
df_fire_sebelum_filter = df_fire.copy()

KALIM_LAT_MIN, KALIM_LAT_MAX = -4.5, 4.5
KALIM_LON_MIN, KALIM_LON_MAX = 108.0, 119.5

outside = df_fire[
    ~((df_fire['latitude'].between(KALIM_LAT_MIN, KALIM_LAT_MAX)) & 
      (df_fire['longitude'].between(KALIM_LON_MIN, KALIM_LON_MAX)))
]

if len(outside) > 0:
    display(Markdown(f"**Ada {len(outside):,} titik yang dihapus karena di luar batas Kalimantan.**"))
    display(outside.head(3))
    
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Plot batas provinsi Kalimantan sebagai basemap asli
    gdf_prov_kalimantan.boundary.plot(ax=ax, linewidth=1, edgecolor='black', zorder=1)
    
    # Titik yang valid
    ax.scatter(df_fire_sebelum_filter['longitude'], df_fire_sebelum_filter['latitude'], s=0.5, c='gray', alpha=0.3, label='Semua deteksi', zorder=2)
    
    # Titik yang dibuang
    ax.scatter(outside['longitude'], outside['latitude'], s=8, c='red', alpha=0.9, label=f'Dibuang ({len(outside)})', zorder=3)
    
    rect = plt.Rectangle((KALIM_LON_MIN, KALIM_LAT_MIN), KALIM_LON_MAX-KALIM_LON_MIN, KALIM_LAT_MAX-KALIM_LAT_MIN,
                          fill=False, edgecolor='blue', linewidth=2, label='Bounding box aktual', zorder=4)
    ax.add_patch(rect)
    ax.set_xlim(107, 121); ax.set_ylim(-5, 8)
    ax.legend(); ax.set_title('Titik yang Dibuang oleh Bounding Box')
    
    os.makedirs(OUTPUTS, exist_ok=True)
    plt.savefig(os.path.join(OUTPUTS, '01_bbox_filter.png'))
    plt.show()

    df_fire = df_fire[
        (df_fire['latitude'].between(KALIM_LAT_MIN, KALIM_LAT_MAX)) & 
        (df_fire['longitude'].between(KALIM_LON_MIN, KALIM_LON_MAX))
    ].reset_index(drop=True)
else:
    display(Markdown("**Semua titik sudah berada di dalam area Kalimantan.**"))

display(Markdown(f"**Total titik valid setelah difilter:** {len(df_fire):,} baris"))

## 4b. Validasi Batas Wilayah Kalimantan (Point-in-Polygon)
Setelah difilter menggunakan batas kotak (bounding box), kita lakukan pemeriksaan tambahan (sanity check) 
untuk melihat berapa banyak titik yang benar-benar jatuh di dalam daratan Kalimantan (berdasarkan poligon batas provinsi), 
dan berapa banyak yang jatuh di luarnya (misalnya di laut pesisir atau pulau sekitar).
Catatan: Kita tidak menghapus titik di luar poligon pada tahap ini karena titik pesisir/batas laut masih valid untuk dianalisis.

In [ ]:
print("\n[2.6b] Validasi Point-in-Polygon dengan Batas Provinsi")
import geopandas as gpd
import matplotlib.pyplot as plt   # <-- add this

gdf_fire_points = gpd.GeoDataFrame(
    df_fire, 
    geometry=gpd.points_from_xy(df_fire.longitude, df_fire.latitude),
    crs='EPSG:4326'
)

# Gunakan data batas provinsi yang sudah diload sebelumnya
kalimantan_geom = gdf_prov_kalimantan.geometry.make_valid().buffer(0).union_all()
gdf_fire_points['in_kalimantan'] = gdf_fire_points.geometry.within(kalimantan_geom)

pct_inside = gdf_fire_points['in_kalimantan'].mean() * 100
n_outside = (~gdf_fire_points['in_kalimantan']).sum()

display(Markdown(f"**Interpretasi batas wilayah:** **{pct_inside:.2f}%** deteksi benar-benar berada di dalam poligon Kalimantan. Ada **{n_outside:,}** titik yang berada di luar poligon (kemungkinan di pesisir atau laut). Sanity check ini tidak menghapus data."))

fig, ax = plt.subplots(figsize=(10, 8))
gdf_prov_kalimantan.boundary.plot(ax=ax, linewidth=1, edgecolor='black', zorder=2)
gdf_fire_points[gdf_fire_points['in_kalimantan']].plot(ax=ax, markersize=1, alpha=0.3, color='#2b8cbe', label='Di dalam batas', zorder=1)
gdf_fire_points[~gdf_fire_points['in_kalimantan']].plot(ax=ax, markersize=6, alpha=0.7, color='#e34a33', label='Di luar batas', zorder=3)
ax.legend()
ax.set_title('Deteksi Relatif terhadap Poligon Batas Provinsi Kalimantan')
plt.savefig(os.path.join(OUTPUTS, '01_point_in_polygon_check.png'))
plt.show()

# Kembalikan df_fire sebagai DataFrame biasa
df_fire = pd.DataFrame(gdf_fire_points.drop(columns=['geometry', 'in_kalimantan']))

print("\n[2.7] Pemeriksaan kelengkapan waktu")
df_fire['year'] = df_fire['acq_date'].dt.year
df_fire['month'] = df_fire['acq_date'].dt.month

year_counts = df_fire.groupby('year').size()
for y, c in year_counts.items():
    print(f"       Tahun {y}: {c:,} titik panas")

year_end = df_fire.groupby('year')['acq_date'].max()
incomplete_years = year_end[year_end.dt.month < 12].index.tolist()
if incomplete_years:
    display(Markdown(f"**Peringatan Tahun Parsial:** Tahun **{incomplete_years}** tidak memiliki data hingga Desember, sehingga total tahunannya tidak boleh dibandingkan secara langsung dengan tahun yang lengkap."))

## Analisis Anomali Intensitas Kebakaran dan Area Piksel
Memeriksa distribusi nilai Fire Radiative Power (FRP) untuk menemukan kejadian *mega-fire* ekstrem serta menghitung perkiraan luas area yang ditangkap setiap piksel berdasarkan atribut pindaian satelit.

In [ ]:
print("\n[2.8] Analisis anomali FRP")
frp_desc = df_fire['frp'].describe(percentiles=[.25, .5, .75, .9, .95, .99])
print(frp_desc)
print(f"\n     Rentang FRP: {df_fire['frp'].min():.2f} hingga {df_fire['frp'].max():.2f} MW")

top_01 = df_fire['frp'].quantile(0.999)
mega_fires = df_fire[df_fire['frp'] >= top_01]
print(f"     Ambang batas persentil 99.9: {top_01:.2f} MW")
print(f"     Kejadian ekstrem di atas ambang batas: {len(mega_fires):,} titik")

print("\n[2.9] Perhitungan area piksel")
df_fire['pixel_area_km2'] = df_fire['scan'] * df_fire['track']
print(f"     Rata rata area piksel: {df_fire['pixel_area_km2'].mean():.4f} kilometer persegi")

## 5. Ekstraksi Fitur Waktu dan Simpan Data
Terakhir untuk Fase 1, kita ekstrak nama hari dan minggu dari tanggal untuk mempermudah analisis nanti. 
Setelah semua beres, kita simpan data yang sudah bersih ini ke folder `processed` supaya siap dipakai di fase selanjutnya.

In [ ]:
df_fire['day_of_week'] = df_fire['datetime_wib'].dt.day_name()
df_fire['week'] = df_fire['datetime_wib'].dt.isocalendar().week.astype(int)

output_path = os.path.join(DATA_PROCESSED, 'hotspot_cleaned.csv')
df_fire.to_csv(output_path, index=False)
display(Markdown(f"**Data bersih sudah diamankan di:** `{output_path}`"))
display(df_fire.head(3))